# Lesson 04: Reranking and the threshold (solution)

**Goal:** re-sort retrieved chunks with a reranker and see what a relevance threshold does.

**Before you start**
- Theory: [lesson 04](../../../docs/lessons/04_rerank_and_threshold.md)
- Python concepts: `sorted(..., key=lambda ...)`, `zip`, dict comprehensions
- Files used: `fixtures/tiny.pdf`, the fake embedder and reranker, Qdrant in-memory. Offline, no API key, no cost.

Each step: read the short explanation, fill in the `# TODO` cell, run the check cell below it.

## Setup (just run it)

In [ ]:
import warnings
from pathlib import Path

from docrag.embed.base import EmbeddingService
from docrag.embed.registry import make_embedder
from docrag.ingest.pipeline import ingest_pdf
from docrag.rerank.reranker import FakeReranker, calibrate_threshold, rerank
from docrag.store.qdrant import QdrantStore, make_client

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)
chunks = ingest_pdf(ROOT / "fixtures" / "tiny.pdf", "tiny")
# In-memory Qdrant warns that payload indexes do nothing locally; expected here.
warnings.filterwarnings("ignore", message="Payload indexes have no effect")
print(len(chunks), "chunks:", [c.id for c in chunks])

embeddings = EmbeddingService(make_embedder("fake:64", ROOT / "models" / "hf"))
store = QdrantStore(make_client(":memory:"), "lesson04", embeddings.dim)
store.ensure_collection()
store.upsert(chunks, embeddings.embed([c.embed_text for c in chunks]))

q = "What does the report recommend for Asia?"
hits = store.search(
    query_vector=embeddings.embed_query(q), query_text=q, mode="hybrid", limit=10
)
reranker = FakeReranker()  # word overlap; the real one is a cross-encoder (ONNX)
print("hybrid order:", [h.chunk.id for h in hits])

## Step 1: Score and sort

A reranker reads the question and each chunk *together* and returns one relevance score per chunk: `reranker.score(q, texts)`. Pair ids with scores using `zip(..., strict=True)` (strict: fail loudly if the lengths differ), then sort with `sorted(pairs, key=lambda p: p[1], reverse=True)` (`lambda p: p[1]` is `p => p[1]` in TS).

In [ ]:
scores = reranker.score(q, [h.chunk.text for h in hits])
ranked = sorted(
    zip([h.chunk.id for h in hits], scores, strict=True),
    key=lambda p: p[1],
    reverse=True,
)

In [ ]:
assert len(scores) == len(hits), "One score per hit"
assert [s for _, s in ranked] == sorted(scores, reverse=True), (
    "Sort by score, descending"
)
assert ranked[0][0] == "tiny:p3:c0", "The conclusion page talks about Asia"
for cid, s in ranked:
    print(f"{s:.3f}  {cid}")
print("OK")

## Step 2: What a threshold keeps

`rerank(reranker, q, hits, top_k=10, threshold=t)` drops hits scoring below `t`. Count how many survive for each threshold with a dict comprehension: `{t: ... for t in thresholds}`.

In [ ]:
thresholds = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
survivors = {
    t: len(rerank(reranker, q, hits, top_k=10, threshold=t)) for t in thresholds
}

In [ ]:
assert list(survivors) == thresholds, "One entry per threshold"
counts = list(survivors.values())
assert counts == sorted(counts, reverse=True), (
    "A higher threshold can only keep fewer hits"
)
assert survivors[0.0] == len(hits)
print(survivors)
print("OK")

## Step 3: Calibrate the threshold

The pipeline does **not** filter hits by the threshold (that destroyed recall in the first run). It uses it as a gate: "is even the *best* chunk relevant enough to answer?" `calibrate_threshold(scores, labels)` picks the cut that maximizes F1 of "score >= t means answerable", on the dev split only. Try it on four toy questions.

In [ ]:
best_scores = [0.9, 0.8, 0.3, 0.2]  # best rerank score per question
should_answer = [True, True, False, False]
threshold, f1 = calibrate_threshold(best_scores, should_answer)

In [ ]:
assert threshold is not None, "Hint: calibrate_threshold(best_scores, should_answer)"
assert (threshold, f1) == (0.8, 1.0), (
    "The cut sits at the lowest score that should still be answered"
)
print(f"threshold={threshold}, F1={f1}")
print("OK")

## Observation question

What does a threshold that's too high cost?

<details>
<summary>Answer</summary>

The gate fires more often: the system abstains on questions it could have answered, so "correct" on answerable questions falls while "correct" on unanswerable ones rises. That's why the threshold is calibrated on dev (never on test) and reported together with the abstention rate. The abstention experiment showed something less obvious, though: with the calibrated gate, only about 2% of answerable dev questions were stopped before the LLM. Most abstentions came from the LLM itself, so the fix was the prompt and the context size, not the threshold.
</details>

## The real code

- Reranker, rerank(), calibrate_threshold: [`src/docrag/rerank/reranker.py`](../../../src/docrag/rerank/reranker.py)
- Calibration on dev: [`src/docrag/eval/calibrate.py`](../../../src/docrag/eval/calibrate.py)
- The gate: [`src/docrag/answer/service.py`](../../../src/docrag/answer/service.py)
- Lesson: [`docs/lessons/04_rerank_and_threshold.md`](../../../docs/lessons/04_rerank_and_threshold.md)